In [1]:
from pathlib import Path
import zipfile
import pandas as pd

pd.set_option("display.float_format", "{:,.1f}".format)
CARTELLA_GREZZI = Path("../dati_grezzi")
CARTELLA_OMI = CARTELLA_GREZZI / "omi"
CARTELLA_PULITI = Path("../dati_puliti")
CARTELLA_PULITI.mkdir(exist_ok=True)

file_zip = sorted(CARTELLA_OMI.glob("*.zip"))
print("File zip trovati:", len(file_zip))

File zip trovati: 12


In [2]:
def leggi_valori_omi(percorso_zip):
    """Apre uno zip OMI e legge il file VALORI, senza bisogno di estrarlo."""
    with zipfile.ZipFile(percorso_zip) as z:
        nome = [n for n in z.namelist() if n.upper().endswith("VALORI.CSV")][0]
        with z.open(nome) as f:
            return pd.read_csv(
                f, sep=";", skiprows=1, encoding="latin1",
                dtype=str, keep_default_na=False, na_values=["", " "],
            )

omi = pd.concat([leggi_valori_omi(z) for z in file_zip], ignore_index=True)
omi = omi.loc[:, ~omi.columns.str.startswith("Unnamed")]

print("Righe totali:", len(omi))
print("Righe per provincia:")
print(omi["Prov"].value_counts())

Righe totali: 26312
Righe per provincia:
Prov
MI    3773
BG    3522
LE    3182
SA    2923
NA    2426
RM    2341
BA    2084
FR    1996
PV    1591
LT    1122
FG     685
CE     667
Name: count, dtype: int64


In [3]:
omi.groupby("Descr_Tipologia").size().sort_values(ascending=False)

Descr_Tipologia
Abitazioni civili                4652
Abitazioni di tipo economico     3136
Negozi                           3125
Ville e Villini                  3097
Box                              3084
Magazzini                        2004
Uffici                           1932
Laboratori                       1684
Capannoni industriali             991
Capannoni tipici                  919
Posti auto coperti                506
Posti auto scoperti               445
Autorimesse                       441
Uffici strutturati                147
Abitazioni signorili               74
Centri commerciali                 74
Abitazioni tipiche dei luoghi       1
dtype: int64

In [4]:
case = omi[(omi["Descr_Tipologia"] == "Abitazioni civili") & (omi["Stato"] == "NORMALE")].copy()

# I numeri usano la virgola decimale (es. "8,8"): la sostituiamo col punto e convertiamo
for col in ["Compr_min", "Compr_max", "Loc_min", "Loc_max"]:
    case[col] = pd.to_numeric(case[col].str.replace(",", "."), errors="coerce")

case["prezzo_mq"] = (case["Compr_min"] + case["Compr_max"]) / 2
case["affitto_mq_mese"] = (case["Loc_min"] + case["Loc_max"]) / 2

print("Zone con abitazioni civili in stato normale:", len(case))
print("Zone senza dato di affitto:", case["affitto_mq_mese"].isna().sum())
case[["Prov", "Comune_descrizione", "Zona", "Compr_min", "Compr_max", "prezzo_mq", "affitto_mq_mese"]].head()

Zone con abitazioni civili in stato normale: 3747
Zone senza dato di affitto: 351


,Prov,Comune_descrizione,Zona,Compr_min,Compr_max,prezzo_mq,affitto_mq_mese
1,SA,SALERNO,B10,2850,4000,"3,425.0",8.9
9,SA,SALERNO,B11,2100,2800,"2,450.0",6.8
15,SA,SALERNO,B13,2100,2600,"2,350.0",6.9
22,SA,SALERNO,C1,2150,2450,"2,300.0",7.1
31,SA,SALERNO,C2,1800,2250,"2,025.0",6.2


In [5]:
comuni = case.groupby(["Comune_amm", "Comune_descrizione"]).agg(
    prov_omi=("Prov", "first"),
    n_zone=("Zona", "count"),
    prezzo_mq=("prezzo_mq", "mean"),
    affitto_mq_mese=("affitto_mq_mese", "mean"),
).reset_index()
print("Comuni con quotazioni:", len(comuni))
comuni.head()

Comuni con quotazioni: 1401


,Comune_amm,Comune_descrizione,prov_omi,n_zone,prezzo_mq,affitto_mq_mese
0,A010,ABBIATEGRASSO,MI,3,"1,566.7",6.9
1,A015,ACCADIA,FG,1,425.0,1.7
2,A023,ACERNO,SA,2,740.0,2.0
3,A024,ACERRA,NA,5,"1,311.0",4.4
4,A032,ACQUAFONDATA,FR,2,398.8,2.5


In [6]:
file_mef = sorted(CARTELLA_GREZZI.glob("Redditi_e_principali_variabili_IRPEF*comunale*.csv"))[0]
mef = pd.read_csv(
    file_mef, sep=";", encoding="latin1", dtype=str,
    keep_default_na=False, na_values=[""],
    usecols=["Codice catastale", "Codice Istat Comune", "Sigla Provincia", "Numero contribuenti"],
).rename(columns={
    "Codice catastale": "Comune_amm",
    "Codice Istat Comune": "codice_comune",
    "Sigla Provincia": "sigla_attuale",
    "Numero contribuenti": "peso",
})
mef["peso"] = mef["peso"].astype(int)
mef["codice_provincia"] = mef["codice_comune"].str[:3]

comuni = comuni.merge(mef, on="Comune_amm", how="left")
print("Comuni senza corrispondenza nel MEF:", comuni["codice_comune"].isna().sum())

# Confronto: provincia secondo l'OMI vs provincia attuale
pd.crosstab(comuni["prov_omi"], comuni["sigla_attuale"])

Comuni senza corrispondenza nel MEF: 0


sigla_attuale,BA,BG,BT,CE,FG,FR,LE,LT,MB,MI,NA,PV,RM,SA
prov_omi,,,,,,,,,,,,,,
BA,41,0,7,0,0,0,0,0,0,0,0,0,0,0
BG,0,243,0,0,0,0,0,0,0,0,0,0,0,0
CE,0,0,0,104,0,0,0,0,0,0,0,0,0,0
FG,0,0,3,0,58,0,0,0,0,0,0,0,0,0
FR,0,0,0,0,0,91,0,0,0,0,0,0,0,0
LE,0,0,0,0,0,0,96,0,0,0,0,0,0,0
LT,0,0,0,0,0,0,0,33,0,0,0,0,0,0
MI,0,0,0,0,0,0,0,0,55,133,0,0,0,0
NA,0,0,0,0,0,0,0,0,0,0,92,0,0,0


Teniamo solo i comuni che **oggi** appartengono alle nostre 12 province (così i confini sono gli stessi dei dati Istat e MEF).

In [7]:
PROVINCE = {
    "015": "Milano", "016": "Bergamo", "018": "Pavia",
    "058": "Roma", "059": "Latina", "060": "Frosinone",
    "061": "Caserta", "063": "Napoli", "065": "Salerno",
    "071": "Foggia", "072": "Bari", "075": "Lecce",
}
comuni = comuni[comuni["codice_provincia"].isin(PROVINCE)]
print("Comuni nelle 12 province attuali:", len(comuni))
print(comuni["sigla_attuale"].value_counts())

Comuni nelle 12 province attuali: 1336
sigla_attuale
BG    243
PV    184
SA    140
MI    133
RM    121
CE    104
LE     96
NA     92
FR     91
FG     58
BA     41
LT     33
Name: count, dtype: int64


In [8]:
def media_pesata(gruppo, colonna):
    """Media di una colonna pesata per il numero di contribuenti, ignorando i valori mancanti."""
    validi = gruppo.dropna(subset=[colonna])
    return (validi[colonna] * validi["peso"]).sum() / validi["peso"].sum()

righe = []
for codice, gruppo in comuni.groupby("codice_provincia"):
    righe.append({
        "codice_provincia": codice,
        "provincia": PROVINCE[codice],
        "n_comuni_omi": len(gruppo),
        "prezzo_mq_medio": media_pesata(gruppo, "prezzo_mq"),
        "affitto_mq_mese_medio": media_pesata(gruppo, "affitto_mq_mese"),
    })
prezzi = pd.DataFrame(righe)

# Valori del solo comune capoluogo, utili per il confronto
CAPOLUOGHI = ["015146", "016024", "018110", "058091", "059011", "060038",
              "061022", "063049", "065116", "071024", "072006", "075035"]
capoluoghi = comuni[comuni["codice_comune"].isin(CAPOLUOGHI)][
    ["codice_provincia", "prezzo_mq", "affitto_mq_mese"]
].rename(columns={"prezzo_mq": "prezzo_mq_capoluogo",
                  "affitto_mq_mese": "affitto_mq_mese_capoluogo"})
prezzi = prezzi.merge(capoluoghi, on="codice_provincia", how="left")

prezzi["semestre"] = "2026/1"
prezzi = prezzi.round(1)
prezzi.sort_values("prezzo_mq_medio", ascending=False)

,codice_provincia,provincia,n_comuni_omi,prezzo_mq_medio,affitto_mq_mese_medio,prezzo_mq_capoluogo,affitto_mq_mese_capoluogo,semestre
0,015,Milano,133,"3,181.5",11.4,"4,953.1",16.4,2026/1
3,058,Roma,121,"2,566.9",10.5,"3,097.7",12.4,2026/1
7,063,Napoli,92,"1,932.3",6.3,"2,403.3",8.0,2026/1
10,072,Bari,41,"1,400.3",4.9,"1,786.5",6.5,2026/1
4,059,Latina,33,"1,328.8",6.0,"1,444.3",6.3,2026/1
1,016,Bergamo,243,"1,306.8",5.8,"2,043.2",8.8,2026/1
8,065,Salerno,140,"1,218.7",3.5,"1,860.0",5.2,2026/1
2,018,Pavia,184,"1,090.1",4.2,"1,828.6",6.5,2026/1
9,071,Foggia,58,999.9,4.0,"1,193.8",5.5,2026/1
6,061,Caserta,104,883.2,2.8,"1,220.0",3.8,2026/1


In [9]:
assert len(prezzi) == 12, "Manca qualche provincia!"
assert prezzi["prezzo_mq_medio"].notna().all(), "C'è un prezzo mancante!"
print("Controlli superati ✅")

prezzi.to_csv(CARTELLA_PULITI / "omi_prezzi_province.csv", index=False)
print("Salvato in:", CARTELLA_PULITI / "omi_prezzi_province.csv")

Controlli superati ✅
Salvato in: ..\dati_puliti\omi_prezzi_province.csv
